# MolHIV: Vanilla GINE + GSAT
This notebook trains **Vanilla GINE with GSAT** on the **MolHIV (MoleculeNet HIV)** dataset.

Backbone design follows the Vanilla GINE used in `bcos_molhiv.ipynb` (linear node/edge projections, stacked `GINEConv` blocks, batch norm, ReLU, `SumAggregation`, and linear head).

In [1]:
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "bcosgnn").is_dir():
            return p
    raise RuntimeError("Could not locate repo root (pyproject.toml + bcosgnn/).")

current_dir = Path.cwd()
project_root = find_repo_root(current_dir)
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")

Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [2]:
import copy
import random
from dataclasses import dataclass
from typing import Iterable, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch_geometric.datasets import MoleculeNet
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.nn.aggr import SumAggregation

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.9.1
CUDA available: False


In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

@dataclass
class TrainConfig:
    hidden_dim: int = 128
    num_layers: int = 4
    dropout: float = 0.2
    batch_size: int = 64
    max_epochs: int = 80
    lr: float = 1e-3
    weight_decay: float = 1e-5
    early_stop_patience: int = 15
    temperature_start: float = 1.0
    temperature_end: float = 0.1
    gsat_r: float = 0.7
    pred_loss_coef: float = 1.0
    info_loss_coef: float = 1.0

cfg = TrainConfig()
SEEDS = (0, 1, 2)
TEST_SIZE = 0.2
VAL_SIZE = 0.1
DATASET_ROOT = str(project_root / "data" / "MolHIV")

print(cfg)

TrainConfig(hidden_dim=128, num_layers=4, dropout=0.2, batch_size=64, max_epochs=80, lr=0.001, weight_decay=1e-05, early_stop_patience=15, temperature_start=1.0, temperature_end=0.1, gsat_r=0.7, pred_loss_coef=1.0, info_loss_coef=1.0)


In [4]:
def _as_binary_label(y_tensor: torch.Tensor) -> int:
    y = float(y_tensor.view(-1)[0].item())
    return int(y > 0.0)

def load_molhiv_dataset(root: str):
    dataset = MoleculeNet(root=root, name="HIV")
    filtered = []
    for data in dataset:
        y = data.y.view(-1)[0]
        if torch.isfinite(y):
            data.y = torch.tensor([_as_binary_label(data.y)], dtype=torch.long)
            filtered.append(data)
    return filtered

def stratified_split_indices(labels: np.ndarray, seed: int, test_size: float, val_size: float):
    all_idx = np.arange(len(labels))
    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(sss_test.split(all_idx, labels))

    labels_trainval = labels[trainval_idx]
    rel_val_size = val_size / (1.0 - test_size)
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=rel_val_size, random_state=seed)
    train_rel, val_rel = next(sss_val.split(np.zeros_like(labels_trainval), labels_trainval))

    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel]
    return train_idx, val_idx, test_idx

dataset = load_molhiv_dataset(DATASET_ROOT)
labels = np.array([int(d.y.item()) for d in dataset], dtype=np.int64)

print(f"Loaded MolHIV samples: {len(dataset)}")
print(f"Positive ratio: {labels.mean():.4f}")
print(f"Node feature dim: {dataset[0].x.size(-1)} | Edge feature dim: {dataset[0].edge_attr.size(-1)}")

Loaded MolHIV samples: 41120
Positive ratio: 0.0351
Node feature dim: 9 | Edge feature dim: 3


In [5]:
class MaskableGINEConv(MessagePassing):
    """GINEConv that supports edge masking via edge_weight parameter."""
    def __init__(self, nn_mlp, train_eps=False):
        super().__init__(aggr='add') 
        self.nn = nn_mlp
        self.initial_eps = 0.0
        if train_eps:
            self.initial_eps = torch.nn.Parameter(torch.Tensor([0.0]))
            
    def forward(self, x, edge_index, edge_attr, edge_weight=None):
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr, edge_weight=edge_weight)
        x_r = x[1] if isinstance(x, tuple) else x
        out = out + (1 + self.initial_eps) * x_r
        return self.nn(out)

    def message(self, x_j, edge_attr, edge_weight):
        msg = F.relu(x_j + edge_attr)
        if edge_weight is not None:
            msg = msg * edge_weight.view(-1, 1)
        return msg


class VanillaGINEBackbone(nn.Module):
    def __init__(self, node_dim: int, edge_dim: int, hidden_dim: int = 128, num_layers: int = 4, dropout: float = 0.2):
        super().__init__()
        self.node_proj = nn.Linear(node_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(MaskableGINEConv(mlp, train_eps=False))
            self.norms.append(nn.BatchNorm1d(hidden_dim))
        self.agg = SumAggregation()
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr, batch, edge_weight=None):
        x = x.float()
        edge_attr = edge_attr.float()
        x = self.node_proj(x)
        edge_attr = self.edge_proj(edge_attr)

        for conv, bn in zip(self.convs, self.norms):
            x = conv(x, edge_index, edge_attr, edge_weight=edge_weight)
            x = bn(x)
            x = F.relu(x)

        g = self.agg(x, batch)
        g = self.dropout(g)
        return self.head(g).view(-1)


class GSAT(nn.Module):
    def __init__(self, backbone: nn.Module, node_dim: int, edge_dim: int, hidden_dim: int = 128, temperature: float = 1.0):
        super().__init__()
        self.backbone = backbone
        self.temperature = temperature
        self.att_mlp = nn.Sequential(
            nn.Linear((2 * node_dim) + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        for module in self.att_mlp.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)

    def get_mask(self, x, edge_index, edge_attr, training: bool = True):
        row, col = edge_index
        edge_rep = torch.cat([x[row], x[col], edge_attr], dim=-1)
        edge_logits = self.att_mlp(edge_rep).view(-1)

        if training:
            u = torch.rand_like(edge_logits)
            noise = torch.log(u + 1e-8) - torch.log(1 - u + 1e-8)
            mask = torch.sigmoid((edge_logits + noise) / self.temperature)
        else:
            mask = torch.sigmoid(edge_logits)

        return mask, edge_logits

    def forward(self, data, training: bool = True):
        x = data.x.float()
        edge_index = data.edge_index
        edge_attr = data.edge_attr.float()
        batch = data.batch

        mask, mask_logits = self.get_mask(x, edge_index, edge_attr, training=training)
        pred_logits = self.backbone(x, edge_index, edge_attr, batch, edge_weight=mask)
        return pred_logits, mask, mask_logits

In [6]:
def gsat_loss(pred_logits, labels, mask_logits, r=0.7, pred_loss_coef=1.0, info_loss_coef=1.0):
    labels = labels.view(-1).float()
    pred_loss = F.binary_cross_entropy_with_logits(pred_logits, labels)

    mask_probs = torch.sigmoid(mask_logits)
    prior_target = torch.full_like(mask_probs, 1.0 - r)
    info_loss = F.binary_cross_entropy(mask_probs, prior_target, reduction="mean")

    total_loss = (pred_loss_coef * pred_loss) + (info_loss_coef * info_loss)
    return total_loss, pred_loss, info_loss

def compute_roc_auc_from_logits(logits: List[float], labels: List[int]) -> float:
    y_true = np.asarray(labels, dtype=np.int64)
    y_score = torch.sigmoid(torch.tensor(logits)).numpy()
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def run_epoch_gsat(model, loader, optimizer, device, cfg: TrainConfig, epoch: int, total_epochs: int):
    is_train = optimizer is not None
    model.train(mode=is_train)

    if is_train:
        alpha = (epoch - 1) / max(1, total_epochs - 1)
        model.temperature = cfg.temperature_start + alpha * (cfg.temperature_end - cfg.temperature_start)

    total_loss = 0.0
    total_pred_loss = 0.0
    total_info_loss = 0.0
    logits_all, y_all = [], []

    for batch in loader:
        batch = batch.to(device)
        pred_logits, _, mask_logits = model(batch, training=is_train)
        loss, pred_loss, info_loss = gsat_loss(
            pred_logits=pred_logits,
            labels=batch.y,
            mask_logits=mask_logits,
            r=cfg.gsat_r,
            pred_loss_coef=cfg.pred_loss_coef,
            info_loss_coef=cfg.info_loss_coef,
        )

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        n_graphs = batch.num_graphs
        total_loss += float(loss.item()) * n_graphs
        total_pred_loss += float(pred_loss.item()) * n_graphs
        total_info_loss += float(info_loss.item()) * n_graphs
        logits_all.extend(pred_logits.detach().cpu().tolist())
        y_all.extend(batch.y.view(-1).detach().cpu().tolist())

    n = max(1, len(y_all))
    return {
        "loss": total_loss / n,
        "pred_loss": total_pred_loss / n,
        "info_loss": total_info_loss / n,
        "roc_auc": compute_roc_auc_from_logits(logits_all, y_all),
    }

In [7]:
def train_one_seed_gsat(dataset, labels, cfg: TrainConfig, seed: int, device: torch.device):
    set_seed(seed)

    train_idx, val_idx, test_idx = stratified_split_indices(
        labels=labels,
        seed=seed,
        test_size=TEST_SIZE,
        val_size=VAL_SIZE,
    )

    sample = dataset[0]
    node_dim = int(sample.x.size(-1))
    edge_dim = int(sample.edge_attr.size(-1))

    backbone = VanillaGINEBackbone(
        node_dim=node_dim,
        edge_dim=edge_dim,
        hidden_dim=cfg.hidden_dim,
        num_layers=cfg.num_layers,
        dropout=cfg.dropout,
    )
    model = GSAT(
        backbone=backbone,
        node_dim=node_dim,
        edge_dim=edge_dim,
        hidden_dim=cfg.hidden_dim,
        temperature=cfg.temperature_start,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_loader = DataLoader([dataset[i] for i in train_idx], batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader([dataset[i] for i in val_idx], batch_size=cfg.batch_size, shuffle=False)
    test_loader = DataLoader([dataset[i] for i in test_idx], batch_size=cfg.batch_size, shuffle=False)

    best_state = None
    best_val_auc = -float("inf")
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(1, cfg.max_epochs + 1):
        train_metrics = run_epoch_gsat(model, train_loader, optimizer, device, cfg, epoch, cfg.max_epochs)
        val_metrics = run_epoch_gsat(model, val_loader, optimizer=None, device=device, cfg=cfg, epoch=epoch, total_epochs=cfg.max_epochs)

        val_auc = val_metrics["roc_auc"]
        if np.isnan(val_auc):
            val_auc = -float("inf")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Seed {seed} | Epoch {epoch:03d} | "
                f"train_loss={train_metrics['loss']:.4f} train_auc={train_metrics['roc_auc']:.4f} | "
                f"val_loss={val_metrics['loss']:.4f} val_auc={val_metrics['roc_auc']:.4f}"
            )

        if bad_epochs >= cfg.early_stop_patience:
            print(f"Seed {seed}: early stop at epoch {epoch} (best val epoch={best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_metrics = run_epoch_gsat(model, test_loader, optimizer=None, device=device, cfg=cfg, epoch=cfg.max_epochs, total_epochs=cfg.max_epochs)

    return {
        "seed": int(seed),
        "best_val_epoch": int(best_epoch),
        "best_val_roc_auc": float(best_val_auc),
        "test_roc_auc": float(test_metrics["roc_auc"]),
        "test_loss": float(test_metrics["loss"]),
    }

def run_gsat_molhiv_experiment(dataset, labels, cfg: TrainConfig, seeds: Iterable[int]):
    device = get_device()
    print(f"Running Vanilla GINE + GSAT on device: {device}")

    rows = []
    for seed in seeds:
        out = train_one_seed_gsat(dataset, labels, cfg, int(seed), device)
        rows.append(out)
        print(
            f"Seed {seed} | test_auc={out['test_roc_auc']:.4f} | "
            f"best_val_epoch={out['best_val_epoch']} | best_val_auc={out['best_val_roc_auc']:.4f}"
        )

    results_df = pd.DataFrame(rows).sort_values("seed").reset_index(drop=True)
    summary_df = pd.DataFrame([
        {
            "model": "Vanilla GINE + GSAT",
            "n_seeds": int(len(results_df)),
            "test_roc_auc_mean": float(results_df["test_roc_auc"].mean()),
            "test_roc_auc_std": float(results_df["test_roc_auc"].std(ddof=1)) if len(results_df) > 1 else 0.0,
            "best_val_epoch_mean": float(results_df["best_val_epoch"].mean()),
            "best_val_epoch_std": float(results_df["best_val_epoch"].std(ddof=1)) if len(results_df) > 1 else 0.0,
        }
    ])
    return results_df, summary_df

In [8]:
results_df, summary_df = run_gsat_molhiv_experiment(
    dataset=dataset,
    labels=labels,
    cfg=cfg,
    seeds=SEEDS,
)

display(results_df)
display(summary_df)

print("\nCompact report:")
row = summary_df.iloc[0]
print(
    f"{row['model']}: test ROC-AUC = {row['test_roc_auc_mean']:.4f} ± {row['test_roc_auc_std']:.4f}; "
    f"best val epoch = {row['best_val_epoch_mean']:.2f} ± {row['best_val_epoch_std']:.2f}"
)

Running Vanilla GINE + GSAT on device: cpu
Seed 0 | Epoch 001 | train_loss=0.9269 train_auc=0.4295 | val_loss=0.8290 val_auc=0.4376
Seed 0 | Epoch 001 | train_loss=0.9269 train_auc=0.4295 | val_loss=0.8290 val_auc=0.4376
Seed 0 | Epoch 010 | train_loss=0.7644 train_auc=0.6157 | val_loss=0.7824 val_auc=0.6624
Seed 0 | Epoch 010 | train_loss=0.7644 train_auc=0.6157 | val_loss=0.7824 val_auc=0.6624
Seed 0 | Epoch 020 | train_loss=0.7502 train_auc=0.6845 | val_loss=0.7644 val_auc=0.6952
Seed 0 | Epoch 020 | train_loss=0.7502 train_auc=0.6845 | val_loss=0.7644 val_auc=0.6952
Seed 0 | Epoch 030 | train_loss=0.7457 train_auc=0.7205 | val_loss=0.7480 val_auc=0.7139
Seed 0 | Epoch 030 | train_loss=0.7457 train_auc=0.7205 | val_loss=0.7480 val_auc=0.7139
Seed 0 | Epoch 040 | train_loss=0.7431 train_auc=0.7335 | val_loss=0.7440 val_auc=0.7309
Seed 0 | Epoch 040 | train_loss=0.7431 train_auc=0.7335 | val_loss=0.7440 val_auc=0.7309
Seed 0 | Epoch 050 | train_loss=0.7409 train_auc=0.7401 | val_loss=

,seed,best_val_epoch,best_val_roc_auc,test_roc_auc,test_loss
0,0,50,0.733663,0.747105,0.741859
1,1,64,0.783908,0.740300,0.746934
2,2,45,0.786521,0.774748,0.738507


,model,n_seeds,test_roc_auc_mean,test_roc_auc_std,best_val_epoch_mean,best_val_epoch_std
0,Vanilla GINE + GSAT,3,0.754051,0.018244,53.0,9.848858



Compact report:
Vanilla GINE + GSAT: test ROC-AUC = 0.7541 ± 0.0182; best val epoch = 53.00 ± 9.85
